# Demo multicorpus

Integra `MuseTrainer/library`, `SymbTr`, `PDMX` y `JAZZMUS` dentro del pipeline armonico del proyecto.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / 'src').exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / 'src').exists():
            PROJECT_ROOT = parent
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
INCLUDE_LIBRARY = PROJECT_ROOT / 'external' / 'library' / 'scores'
INCLUDE_SYMBTR = PROJECT_ROOT / 'external' / 'SymbTr'
INCLUDE_JAZZMUS = None  # Ejemplo: PROJECT_ROOT / 'data' / 'jazzmus_dataset'
INCLUDE_PDMX = None     # Ejemplo: Path('/ruta/local/a/PDMX_dataset')

MODEL = 'finite_hmm'    # 'finite_hmm', 'hdp_hmm' o 'both'
OBS = 'pitch_class'
SAMPLE_PER_SOURCE = 5
LIMIT = 10
K = 12
ITERS = 30
BURN_IN = 15
OUTPUT_DIR = PROJECT_ROOT / 'artifacts' / 'outputs' / 'demo_multicorpus'


In [ ]:
from src.analysis.library_batch import analyze_multicorpus
from src.data.multicorpus import CorpusSource

sources = []
if INCLUDE_LIBRARY:
    sources.append(CorpusSource(name='MuseTrainer', source_type='generic', root_dir=INCLUDE_LIBRARY))
if INCLUDE_SYMBTR:
    sources.append(CorpusSource(name='SymbTr', source_type='symbtr', root_dir=INCLUDE_SYMBTR))
if INCLUDE_JAZZMUS:
    sources.append(CorpusSource(name='JAZZMUS', source_type='jazzmus', root_dir=INCLUDE_JAZZMUS))
if INCLUDE_PDMX:
    sources.append(CorpusSource(name='PDMX', source_type='pdmx', root_dir=INCLUDE_PDMX))

outputs = analyze_multicorpus(
    sources=sources,
    output_dir=OUTPUT_DIR,
    obs_type=OBS,
    model=MODEL,
    limit=LIMIT,
    file_limit_per_source=SAMPLE_PER_SOURCE,
    hdp_params={'n_states': K, 'n_iters': ITERS, 'burn_in': BURN_IN, 'seed': 7},
)
outputs

In [ ]:
import pandas as pd

catalog = pd.read_csv(OUTPUT_DIR / 'catalog' / 'catalog.csv')
analysis = pd.read_csv(OUTPUT_DIR / 'analysis' / 'analysis.csv')

print('Obras catalogadas:', len(catalog))
print('Obras analizadas:', len(analysis))
display(catalog[['source_name', 'source_type', 'title', 'composer', 'genre_family', 'style_system']].head(20))
display(analysis.head(20))

In [ ]:
from IPython.display import Image, display

figure_paths = [
    OUTPUT_DIR / 'catalog' / 'figures' / 'catalog_by_source.png',
    OUTPUT_DIR / 'catalog' / 'figures' / 'catalog_by_composer.png',
    OUTPUT_DIR / 'analysis' / 'figures' / 'composer_state_complexity.png',
]

for figure_path in figure_paths:
    if figure_path.exists():
        display(Image(filename=str(figure_path)))